## Caso de estudio: comercio electrónico, clientes y operación

**Nombre:** Maria Fernanda Romero

# Fase 1. Comprensión del problema

## 1.1 Conocer el contexto

1. ¿Qué tipo de organización o actividad económica representa este conjunto de datos?
Rta= Es una plataforma/marketplace de comercio electrónico, donde vendedores independientes ofrecen productos y la plataforma gestiona el envío.
2. ¿Qué procesos del negocio crees que pueden estudiarse con esta información?
Rta= Se puede estudiar la gestión de envíos (tiempos entrega, logistica, satisfaccion), ventas en general (ventas por productos, por geografía), entre otros.
3. ¿Qué decisiones podría tomar una empresa a partir de un análisis de estos datos?
Rta= como mejorar tiempos de entrega, zonas que requieran atención por ventas, que zonas aplicar promociones que impulsen ventas, que productos son los más vendidos.
4. ¿Qué información crees que hace falta para comprender completamente el negocio?
Rta= Datos de devoluciones, costos de las líneas de ventas, si existen bonificaciones a vendedores.

## 1.2 Conectar el análisis con una perspectiva profesional

Piensa en tu formación, experiencia o área de interés.

1. ¿Qué aspecto del negocio te resulta más interesante investigar?
Rta= Teniendo en cuenta que soy analista de laboratorio, me enfocaría en los tiempos de entrega y la calidad del sistema de gestión de envíos.
2. ¿Qué problema real de esa área podrías analizar con estos datos?
Rta= Las calificaciones bajas se refieren directamente a los pedidos que se entregan con retraso y como esto afecta la percepcion de la empresa.
3. ¿Qué archivo podría contener información útil para investigarlo?
Rta= olist_orders_dataset.csv , olist_order_reviews_dataset , olist_customers_dataset
4. ¿Qué variable o indicador les gustaría comprender?
Rta= Días de retraso y calificacion de los clientes
5. ¿Por qué el resultado de tu análisis sería útil para alguien que debe tomar decisiones?
Rta= Permite tomar acciones en mejoras logísticas y estimar cuanta satisfacción se pierde por cada día de retraso.

**Define una perspectiva inicial de análisis:**


In [1]:
# Control de calidad del servicio, analizar como influye en la satisfacción del cliente el cumplimiento de los tiempos de entrega dados en la plataforma y ver si hay efecto
# segun la ubicación del cliente


# Fase 2. Selección y conocimiento de los datos

## 2.1 Selección de un único archivo

1. ¿Qué pregunta de negocio quieres responder?
Rta= ¿cómo afecta los retrasos en las entregas de pedidos a la satisfacción del cliente y varía ese efecto según la ubicación geográfica del cliente?
2. ¿Qué archivo o archivos necesitan para responderla?
Rta= Archivo principal - olist_orders_dataset.csv ; archivos complementarios - olist_order_reviews_dataset , olist_customers_dataset
3. ¿Por qué seleccionaron esos archivos y descartaron los demás?
Rta= Porque contienen las variables a analizar; cumplimiento tiempo de entregas, atisfacción del cliente y estados con ciudades de los clientes.
4. ¿Qué representa una fila en cada archivo seleccionado?
Rta= olist_orders_dataset.csv: representa 1 pedido único ; olist_order_reviews_dataset: representa 1 reseña de un pedido ; olist_customers_dataset: 1 cliente por pedido
5. ¿Cuál podría ser la llave o identificador que permita relacionar los archivos?
Rta= Order_id une Orders con Orders_review ; customer_id une orders con customer
6. ¿Qué riesgos existen al unir tablas que tienen diferentes niveles de detalle?
Rta= Eliminar pedidos que no tengan reseñas, pedidos que no se hayan entregado o que no hayan fecha de entrega, informacion concentrada en unas pocas zonas.

**Nombre del único archivo seleccionado y justificación:** Archivo seleccionado: olist_orders_dataset.csv
**Justificación:** Contiene la información base de fechas de entrega y de los pedidos totales.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [3]:
orders = pd.read_csv("olist_orders_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")

## 2.2 Reconocimiento de la estructura

Explora los datos antes de modificarlos.

In [4]:
#1. ¿Cuántas filas y columnas tiene cada DataFrame?
datasets={
    "Ordenes": orders,
    "Resenas": reviews,
    "Clientes": customers,
}

for nombre, df_ in datasets.items():
    print(f"{nombre} tipo: {type(df_).__name__}, "
          f"filas: {df_.shape[0]:,}, columnas: {df_.shape[1]}")

Ordenes tipo: DataFrame, filas: 99,441, columnas: 8
Resenas tipo: DataFrame, filas: 99,224, columnas: 7
Clientes tipo: DataFrame, filas: 99,441, columnas: 5


**Registro de observaciones:**

In [5]:
print("Dimensiones de ordenes:", orders.shape)

display(orders.head())
display(orders.tail())

orders.info()

display(orders.describe())
display(orders.describe(include="object"))
display(orders.dtypes)

Dimensiones de ordenes: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00
99440,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-03-08 20:57:30,2018-03-09 11:20:28,2018-03-09 22:11:59,2018-03-16 13:08:30,2018-04-03 00:00:00


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


C:\Users\maria\AppData\Local\Temp\ipykernel_6880\312462091.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(orders.describe(include="object"))


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [6]:
display(orders.isna().sum().to_frame("valores_faltantes"))

porcentaje_faltantes = (
    orders.isna().mean().mul(100).round(2)
)
display(porcentaje_faltantes.to_frame("porcentaje_faltante"))

,valores_faltantes
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


,porcentaje_faltante
order_id,0.00
customer_id,0.00
order_status,0.00
order_purchase_timestamp,0.00
order_approved_at,0.16
order_delivered_carrier_date,1.79
order_delivered_customer_date,2.98
order_estimated_delivery_date,0.00


In [7]:
print("Duplicados completos:", orders.duplicated().sum())
display(orders["order_status"].value_counts(dropna=False))

Duplicados completos: 0


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

1. ¿Cuántas filas y columnas tiene cada DataFrame?
Rta = Filas 99441 , columnas 8 
2. ¿Qué representa cada columna?
Rta = 
 0   order_id = Identificador unico del pedido   
 1   customer_id  = Identificador del cliente en esa compra  
 2   order_status = Estado del pedido
 3   order_purchase_timestamp = fecha y hora de la compra
 4   order_approved_at = fecha aprobacion del pago
 5   order_delivered_carrier_date = Fecha entregada al transportador
 6   order_delivered_customer_date = fecha de entrega real al cliente 
 7   order_estimated_delivery_date = fecha estimada de entrega al cliente
3. ¿Qué tipos de datos identifica Python?
Rta = Solo identifica como tipo texto, incluido las fechas
4. ¿Qué variables son numéricas, categóricas, temporales o identificadores?
Identificadores= order_id, customer_id ; Categoricas= order_status ; temporales= todas las order_delivered, order_approved y order_estimated
5. ¿Qué variables parecen relevantes para la pregunta que planteaste?
Rta = order_delivered_customer_date, order_estimated_delivery_date, order_status, order_id, customer_id, order_purchase_timestamp

In [8]:
print("Dimensiones de ordenes:", reviews.shape)

display(reviews.head())
display(reviews.tail())

reviews.info()

display(reviews.describe())
display(reviews.describe(include="object"))
display(reviews.dtypes)

Dimensiones de ordenes: (99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13
99223,efe49f1d6f951dd88b51e6ccd4cc548f,90531360ecb1eec2a1fbb265a0db0508,1,NaN,"meu produto chegou e ja tenho que devolver, po...",2017-07-03 00:00:00,2017-07-03 21:01:49


<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


C:\Users\maria\AppData\Local\Temp\ipykernel_6880\376040787.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(reviews.describe(include="object"))


,review_id,order_id,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
count,99224,99224,11568,40977,99224,99224
unique,98410,98673,4527,36159,636,98248
top,c444278834184f72b1484dfe47de7f97,c88b1d1b157a9999ce368f218a407141,Recomendo,Muito bom,2017-12-19 00:00:00,2017-06-15 23:21:05
freq,3,3,423,230,463,4


review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

In [9]:
display(reviews.isna().sum().to_frame("valores_faltantes"))

porcentaje_faltantes = (
    reviews.isna().mean().mul(100).round(2)
)
display(porcentaje_faltantes.to_frame("porcentaje_faltante"))

,valores_faltantes
review_id,0
order_id,0
review_score,0
review_comment_title,87656
review_comment_message,58247
review_creation_date,0
review_answer_timestamp,0


,porcentaje_faltante
review_id,0.00
order_id,0.00
review_score,0.00
review_comment_title,88.34
review_comment_message,58.70
review_creation_date,0.00
review_answer_timestamp,0.00


In [10]:
print("Duplicados completos:", reviews.duplicated().sum())

display(
    reviews[reviews.duplicated(keep=False)]
    .sort_values("order_id")
)

print(
    "Order_id duplicados:",
    reviews["order_id"].duplicated().sum()
)

Duplicados completos: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


Order_id duplicados: 551


In [11]:
display(reviews["review_score"].value_counts(dropna=False))

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

1. ¿Cuántas filas y columnas tiene cada DataFrame?
Rta = Filas 99224 , columnas 7 
2. ¿Qué representa cada columna?
Rta = 
 0   review_id  = identificador de la reseña
 1   order_id  = pedido al que pertenece
 2   review_score  = calificacion
 3   review_comment_title = comentario opcional del cliente
 4   review_comment_message = comentario opcional del cliente
 5   review_creation_date = fecha recibida la encuesta
 6   review_answer_timestamp = fecha y hora de respuesta del cliente
3. ¿Qué tipos de datos identifica Python?
Rta = Identifica como tipo texto e int64
4. ¿Qué variables son numéricas, categóricas, temporales o identificadores?
Identificadores= order_id, review_id ; Numericas= review_score ; Temporales= review_creation_date y review_answer_timestamp
5. ¿Qué variables parecen relevantes para la pregunta que planteaste?
Rta = review_score, order_id

In [12]:
print("Dimensiones de ordenes:", customers.shape)

display(customers.head())
display(customers.tail())

customers.info()

display(customers.describe())
display(customers.describe(include="object"))
display(customers.dtypes)

Dimensiones de ordenes: (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS
99440,274fa6071e5e17fe303b9748641082c8,84732c5050c01db9b23e19ba39899398,6703,cotia,SP


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000


C:\Users\maria\AppData\Local\Temp\ipykernel_6880\366875937.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(customers.describe(include="object"))


,customer_id,customer_unique_id,customer_city,customer_state
count,99441,99441,99441,99441
unique,99441,96096,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,sao paulo,SP
freq,1,17,15540,41746


customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

In [13]:
display(customers.isna().sum().to_frame("valores_faltantes"))

porcentaje_faltantes = (
    customers.isna().mean().mul(100).round(2)
)
display(porcentaje_faltantes.to_frame("porcentaje_faltante"))

,valores_faltantes
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


,porcentaje_faltante
customer_id,0.0
customer_unique_id,0.0
customer_zip_code_prefix,0.0
customer_city,0.0
customer_state,0.0


In [15]:
display(customers["customer_city"].value_counts(dropna=False))
display(customers["customer_state"].value_counts(dropna=False))

customer_city
sao paulo              15540
rio de janeiro          6882
belo horizonte          2773
brasilia                2131
curitiba                1521
                       ...  
siriji                     1
natividade da serra        1
monte bonito               1
sao rafael                 1
eugenio de castro          1
Name: count, Length: 4119, dtype: int64

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

1. ¿Cuántas filas y columnas tiene cada DataFrame?
Rta = Filas 99441 , columnas 5
2. ¿Qué representa cada columna?
Rta = 
 0   customer_id  = identificador del cliente por pedido  
 1   customer_unique_id  = identificador del cliente unico
 2   customer_zip_code_prefix = codigo postal 
 3   customer_city = ciudad del cliente
 4   customer_state = estado del cliente 
3. ¿Qué tipos de datos identifica Python?
Rta = Identifica como tipo texto e int64
4. ¿Qué variables son numéricas, categóricas, temporales o identificadores?
Identificadores= customer_id, customer_unique_id ; Numericas= customer_zip_code_prefix ; categoricas= customer_city, customer_state
5. ¿Qué variables parecen relevantes para la pregunta que planteaste?
Rta = customer_state y customer_id

6. ¿Hay variables cuyo significado no sea evidente? ¿Cómo podrías investigarlo?
Rta= El customer_id y customer_unique_id puede resultar confuso hasta que se revisa la base de datos y se aclara.
order_approved es la aprobacion del pago o del pedido
7. ¿Qué observaciones iniciales te generan curiosidad?
Rta= Hay mas reseñas para algunos pedidos y algunos no cuentan con reseñas. Hay valores nulos en las fechas de entrega. Hay muchos reviews completos sin texto libre. La base
de datos de customers no tiene valores faltantes.

## Antes de unir tablas, analiza

1. ¿Qué representa una fila en cada DataFrame? 
Rta= En orders un pedido, en reviews una reseña y en customer un cliente por pedido
2. ¿Cuál es la llave que relaciona las tablas?
Rta= orders con reviews = order_id ; orders con customer = customer_id
3. ¿La llave es única en una tabla, en ambas o en ninguna?
Rta= La llave orders_id es unica solo en orders, la llave customer_id es unica en ambas tablas
4. ¿La relación es uno a uno, uno a muchos o muchos a muchos?
Rta= Orders a reviews es uno a muchos ya que se vio en el análisis exploratorio que para un pedido pueden haber 2 o mas reseñas. Orders a customer es uno a uno.
5. ¿Qué tipo de unión responde mejor a la pregunta de análisis?
Rta= 
¿cómo afecta los retrasos en las entregas de pedidos a la satisfacción del cliente y varía ese efecto según la ubicación geográfica del cliente?
Es necesario que los pedidos tengan fecha de entrega y calificación, por lo que se usará un inner join entre orders  y reviews. Para customer se usará un left join desde orders, para no perder pedidos.
6. ¿La unión puede duplicar registros? ¿Cómo lo comprobarías?
Si, pasaria al unir orders con reviews porque hay pedidos con mas de una reseña. Por lo que si no se limpia antes, a un mismo pedido se le puede duplicar o triplicar las filas. Con la otra unión se esperaría que no, porque las llaves son unicas.
7. ¿Cuántas filas había antes y cuántas después de unir las tablas?

Original
Ordenes filas: 99,441
Resenas filas: 99,224
Clientes filas: 99,441

Posterior
Limpiando y acomodando las tablas quedaría la union entre orders y reviews serian 99,441, sin reseñas duplicadas. Para orders y customers seria igual 99,441 filas.

## Ejercicio de comprensión (no hace parte obligatoria del EDA)

Imagina que tienes una tabla de pedidos y otra de clientes.

- ¿Qué columna podría relacionarlas?
Rta= La columna del customer_id
- Si quieres conservar todos los pedidos, incluso aquellos que no tengan coincidencia en la tabla de clientes, ¿qué tipo de unión elegirías?
Rta= Se usaría un Left join con pedidos a la izquierda y clientes a la derecha, los pedidos sin coincidencia conservan sus datos y se rellenan con nulos las columnas de clientes
- ¿Qué podría ocurrir si el identificador de cliente aparece varias veces en la tabla de clientes?
Rta= Se duplicaría la cantidad de pedidos de ese cliente respecto a cuantas veces aparezca, por lo que la estadística aumentaría.
- ¿Qué validaciones realizarías después de la unión?
Rta= Comparar el numero de filas después de la unión, verificar que la llave order_id sea unica, revisar los nulos en clientes.

**Explica tus respuestas:**


In [16]:
# Espacio opcional para practicar merge() con dos DataFrames.
# Carga los archivos de pedidos y clientes solo si deseas realizar el ejercicio.
# Escribe aquí tu código de unión y las validaciones posteriores.
print("Filas antes de unión: Orders:", len(orders), " Customer:", len(customers))

pedidos_clientes= orders.merge(
    customers, on="customer_id",
    how="left", validate="one_to_one"
)

print("Filas después de unión: Orders:", len(pedidos_clientes))

print("Validaciones")
print("Filas iguales", len(pedidos_clientes)==len(orders))
print("order_id es unico", pedidos_clientes["order_id"].is_unique)
print("pedidos sin clientes (nulos)", pedidos_clientes["customer_unique_id"].isnull().sum())

pedidos_clientes.head(3)

Filas antes de unión: Orders: 99441  Customer: 99441
Filas después de unión: Orders: 99441
Validaciones
Filas iguales True
order_id es unico True
pedidos sin clientes (nulos) 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO


# Fase 3. Calidad y confiabilidad de los datos

Antes de analizar el comportamiento del negocio, evalúa si los datos son suficientemente confiables.

## 3.1 Diagnóstico

1. ¿Existen valores faltantes? ¿En qué variables se concentran?
Rta= Si, para el análisis se evidencia que hay valores faltantes en orders y especificamente en la variable de order_delivered_customer_date que es de nuestro interés. Así mismo, solo se evidencia valores nulos para reviews en la sección de titulo y mensaje de la reseña. En customers no hay datos nulos.
2. ¿Qué proporción representan los valores faltantes?
Rta= order_approved_at tiene un % de 0,16, order_delivered_carrier_date de 1,79% y order_delivered_customer_date de 2,98%. Los demás no son necesarios.
3. ¿Existen registros duplicados? ¿Son duplicados exactos o podrían representar eventos distintos?
Rta= No hay duplicados exactos. Hay 547 pedidos que tienen más de una reseña.
4. ¿Los tipos de datos corresponden con el significado de las variables?
Rta= No, las columnas de fechas fueron cargadas como texto.
5. ¿Existen valores que parecen imposibles o poco razonables?
Rta= Existen inconsistencias en cuanto a las fechas principalmente, pedidos marcados como entregados pero sin fecha correspondiente.
6. ¿Hay categorías escritas de diferentes maneras que podrían representar lo mismo?
Rta= No se evidencia
7. ¿Los identificadores cumplen alguna condición de unicidad?
Rta= Si, en orders orders_id y customer customer_id, pero en reviews se evidencia order_id y review_id se duplica.
8. ¿Qué otros problemas de calidad pueden afectar sus resultados?
Rta= Se reciben los pedidos canceled, unavailable o shipped dentro de la estadística y afecta este resultado. La calificación puede reflejar la calidad del producto y no solo la entrega. 

**¿Cuál de los problemas encontrados consideras más importante y por qué?**
Rta= El problema importante es la duplicación de las reseñas por pedidos ya que repetirian los resultados estadísticos.

In [17]:
# Escribe aquí el código para investigar los valores faltantes.
display(orders.isna().sum().to_frame("valores_faltantes"))

porcentaje_faltantes = (
    orders.isna().mean().mul(100).round(2)
)
display(porcentaje_faltantes.to_frame("porcentaje_faltante"))

display(reviews.isna().sum().to_frame("valores_faltantes"))

porcentaje_faltantes = (
    reviews.isna().mean().mul(100).round(2)
)
display(porcentaje_faltantes.to_frame("porcentaje_faltante"))

display(customers.isna().sum().to_frame("valores_faltantes"))

porcentaje_faltantes = (
    customers.isna().mean().mul(100).round(2)
)
display(porcentaje_faltantes.to_frame("porcentaje_faltante"))

,valores_faltantes
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


,porcentaje_faltante
order_id,0.00
customer_id,0.00
order_status,0.00
order_purchase_timestamp,0.00
order_approved_at,0.16
order_delivered_carrier_date,1.79
order_delivered_customer_date,2.98
order_estimated_delivery_date,0.00


,valores_faltantes
review_id,0
order_id,0
review_score,0
review_comment_title,87656
review_comment_message,58247
review_creation_date,0
review_answer_timestamp,0


,porcentaje_faltante
review_id,0.00
order_id,0.00
review_score,0.00
review_comment_title,88.34
review_comment_message,58.70
review_creation_date,0.00
review_answer_timestamp,0.00


,valores_faltantes
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


,porcentaje_faltante
customer_id,0.0
customer_unique_id,0.0
customer_zip_code_prefix,0.0
customer_city,0.0
customer_state,0.0


In [18]:
# Escribe aquí el código para investigar duplicados.
print("Duplicados completos:", orders.duplicated().sum())
display(
    orders[orders.duplicated(keep=False)]
    .sort_values("order_id")
)
print(
    "order_id duplicados en ORDERS:",
    orders["order_id"].duplicated().sum()
)

print("\nDuplicados completos:", reviews.duplicated().sum())
display(
    reviews[reviews.duplicated(keep=False)]
    .sort_values("review_id")
)
print(
    "review_id duplicados en REVIEWS:",
    reviews["review_id"].duplicated().sum()
)

print("\nDuplicados completos:", reviews.duplicated().sum())
display(
    reviews[reviews.duplicated(keep=False)]
    .sort_values("order_id")
)
print(
    "order_id duplicados en REVIEWS:",
    reviews["order_id"].duplicated().sum()
)

print("\nDuplicados completos:", customers.duplicated().sum())
display(
    customers[customers.duplicated(keep=False)]
    .sort_values("customer_id")
)
print(
    "customer_id duplicados en CUSTOMER:",
    customers["customer_id"].duplicated().sum()
)

Duplicados completos: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


order_id duplicados en ORDERS: 0

Duplicados completos: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


review_id duplicados en REVIEWS: 814

Duplicados completos: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


order_id duplicados en REVIEWS: 551

Duplicados completos: 0


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


customer_id duplicados en CUSTOMER: 0


In [19]:
# Escribe aquí el código para revisar tipos de datos, categorías y posibles inconsistencias.
objetos = {"orders": orders, "reviews": reviews, "customers": customers}

for nombre, d in objetos.items():
    print(f"{nombre}")
    print(d.dtypes, "\n")

# Categorías: estados y ciudades
print("Estados de pedido:", orders["order_status"].unique())
print("Ciudades:", customers["customer_city"].nunique())

orders
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object 

reviews
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object 

customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object 

Estados de pedido: <StringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str
Ciudades: 4119


In [20]:
# Escribe aquí el código para investigar los valores sospechosos que hayan identificado.
objetos = {"orders": orders, "reviews": reviews, "customers": customers}

for nombre, d in objetos.items():
    nulos = d.isnull().sum()
    resumen = pd.DataFrame({"nulos": nulos,
                            "% nulos": (nulos / len(d) * 100).round(2)})
    print(f"--- {nombre.upper()} ---")
    print(resumen[resumen["nulos"] > 0] if nulos.sum() > 0 else "Sin nulos")
    print()

print("Nulos en fecha de entrega al cliente, por estado del pedido:")
print(orders[orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts())


--- ORDERS ---
                               nulos  % nulos
order_approved_at                160     0.16
order_delivered_carrier_date    1783     1.79
order_delivered_customer_date   2965     2.98



--- REVIEWS ---
                        nulos  % nulos
review_comment_title    87656    88.34
review_comment_message  58247    58.70

--- CUSTOMERS ---
Sin nulos

Nulos en fecha de entrega al cliente, por estado del pedido:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [21]:
# Escribe aquí el código para investigar los valores sospechosos que hayan identificado.
o = orders.copy()
for c in ["order_purchase_timestamp", "order_approved_at",
          "order_delivered_carrier_date", "order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    o[c] = pd.to_datetime(o[c])

# Pedidos delivered pero sin fecha de entrega
print("Delivered sin fecha de entrega:",
      ((orders["order_status"] == "delivered") &
       orders["order_delivered_customer_date"].isnull()).sum())

# Tiempos de entrega extremos
dias = (o["order_delivered_customer_date"] - o["order_purchase_timestamp"]).dt.days
print("\nEntregas de más de 60 días:", (dias > 60).sum())
retraso = (o["order_delivered_customer_date"] - o["order_estimated_delivery_date"]).dt.days
print("Retraso en días:")
print(retraso.describe())
print("Pedidos con retraso (>0):", (retraso > 0).sum(),
      f"({(retraso > 0).mean()*100:.1f} %)")

# Calificaciones fuera del rango 1-5
print("\nCalificaciones fuera de 1-5:",
      (~reviews["review_score"].between(1, 5)).sum())

# Reseñas respondidas antes de crearse
r = reviews.copy()
r["review_creation_date"] = pd.to_datetime(r["review_creation_date"])
r["review_answer_timestamp"] = pd.to_datetime(r["review_answer_timestamp"])
print("\nReseña respondida antes de su creación:",
      (r["review_answer_timestamp"] < r["review_creation_date"]).sum())

Delivered sin fecha de entrega: 8

Entregas de más de 60 días: 288
Retraso en días:
count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
dtype: float64
Pedidos con retraso (>0): 6535 (6.6 %)

Calificaciones fuera de 1-5: 0

Reseña respondida antes de su creación: 0


# 3.2 Decisiones de tratamiento

No todos los problemas deben resolverse de la misma manera.

Para cada problema identificado, responde:

## Problema 1

1. ¿Qué evidencia demuestra que existe un problema?
Rta= reviews-"order_id" tiene 551 repetidos (547 pedidos con 2 o 3 reseñas)
2. ¿Qué alternativas de tratamiento consideran?
Rta= Conservar la reseña más reciente, la más antigua, promedio de calificación del pedido.
3. ¿Qué información se perdería si eliminan registros?
Rta= Se perderían el historial de cambios de opinión y se eliminarían las 551 filas.
4. ¿Qué consecuencias tendría conservar los valores sin modificarlos?
Rta= 551 pedidos pesarían doble o triple en los promedios y el conteo de pedidos se inflaría.
5. ¿Qué decisión tomarán y cómo justificarían esa decisión ante un equipo de negocio?
Rta= Conservar la reseña más reciente por pedido, ya que la última calificación refleja la experiencia final del cliente, es un criterio simple y reproducible, y el impacto es mínimo, si se Promedia crearía notas que el cliente nunca dio.

## Problema 2

1. ¿Qué evidencia demuestra que existe un problema?
Rta= 2,965 nulos en order_delivered_customer_date (2.97 %). De ellos, 2,957 son pedidos no entregados y 8 son pedidos “delivered” sin fecha
2. ¿Qué alternativas de tratamiento consideran?
Rta= Eliminar pedidos sin fecha, poner el valor de la mediana, transformarlos a nulos y excluirlos del cálculo.
3. ¿Qué información se perdería si eliminan registros?
Rta= Se perderían los 2,965 pedidos que en su mayoría son cancelados y no disponibles, que afectan para otros análisis.
4. ¿Qué consecuencias tendría conservar los valores sin modificarlos?
Rta= Quedaría fuera de la estadística
5. ¿Qué decisión tomarán y cómo justificarían esa decisión ante un equipo de negocio?
Rta= Conservar solo los pedidos con order_status == "delivered" y fecha de entrega no nula. Imputar sería inventar un dato clave, y los pedidos no entregados no tienen un retraso medible. La pérdida es pequeña (~3 %).

## Problema 3

1. ¿Qué evidencia demuestra que existe un problema?
Rta= 7 columnas de fecha con dtype str (5 en orders, 2 en reviews).
2. ¿Qué alternativas de tratamiento consideran?
Rta= Convertirlas a datetime o dejarlas como texto.
3. ¿Qué información se perdería si eliminan registros?
Rta= En teoría ninguno ya que no se debe eliminar registros, todo esta completo.
4. ¿Qué consecuencias tendría conservar los valores sin modificarlos?
Rta= No se pueden restar fechas, y es imposible calcular los días de retraso.
5. ¿Qué decisión tomarán y cómo justificarían esa decisión ante un equipo de negocio?
Rta= Convertir a datetime las 5 fechas de orders y review_creation_date (necesaria para ordenar y elegir la reseña más reciente).

In [22]:
# Trabajamos sobre copias para no alterar los DataFrames originales
orders_l = orders.copy()
reviews_l = reviews.copy()
customers_l = customers.copy()

# 1. Convertir fechas a datetime (Problema 3)
cols_orders = ["order_purchase_timestamp", "order_approved_at",
               "order_delivered_carrier_date", "order_delivered_customer_date",
               "order_estimated_delivery_date"]
for c in cols_orders:
    orders_l[c] = pd.to_datetime(orders_l[c])

reviews_l["review_creation_date"] = pd.to_datetime(reviews_l["review_creation_date"])
reviews_l["review_answer_timestamp"] = pd.to_datetime(reviews_l["review_answer_timestamp"])

# 2. Una reseña por pedido: la más reciente (Problema 1)
# Se ordena por fecha de creación y, en caso de empate, por fecha de respuesta
reviews_l = (reviews_l
             .sort_values(["review_creation_date", "review_answer_timestamp"])
             .drop_duplicates(subset="order_id", keep="last"))

# 3. Solo pedidos entregados y con fecha real de entrega (Problema 2)
orders_l = orders_l[orders_l["order_status"] == "delivered"]
orders_l = orders_l.dropna(subset=["order_delivered_customer_date"])

# 4. Unir (validate avisa si la relación no es la esperada)
df = orders_l.merge(reviews_l, on="order_id", how="inner", validate="one_to_one")
df = df.merge(customers_l, on="customer_id", how="left", validate="one_to_one")

# 5. Variable principal: días de retraso (positivo = llegó tarde)
df["dias_retraso"] = (df["order_delivered_customer_date"]
                      - df["order_estimated_delivery_date"]).dt.days

In [ ]:
print("FILAS")
print("orders original:        ", len(orders))
print("orders entregados c/fecha:", len(orders_l))
print("reviews original:       ", len(reviews))
print("reviews una por pedido: ", len(reviews_l))
print("DataFrame final (df):   ", len(df))

print("\nVALIDACIONES")
print("order_id único?         ", df["order_id"].is_unique)
print("Solo pedidos 'delivered'?", (df["order_status"] == "delivered").all())
print("Nulos en fecha de entrega:", df["order_delivered_customer_date"].isnull().sum())
print("Nulos en review_score:    ", df["review_score"].isnull().sum())
print("Nulos en customer_state:  ", df["customer_state"].isnull().sum())

print("\nTIPOS DE FECHA")
print(df[["order_purchase_timestamp", "order_delivered_customer_date",
          "order_estimated_delivery_date", "review_creation_date"]].dtypes)

print("\nPÉRDIDA DE DATOS")
print(f"Pedidos conservados: {len(df):,} de {len(orders):,} "
      f"({len(df)/len(orders)*100:.1f} %)")
print(f"Pedidos descartados: {len(orders) - len(df):,}")

print("\nRETRASO")
print(df["dias_retraso"].describe().round(2))
print(f"Pedidos con retraso (>0): {(df['dias_retraso'] > 0).sum():,} "
      f"({(df['dias_retraso'] > 0).mean()*100:.1f} %)")

print("\nCALIFICACIÓN")
print(df["review_score"].value_counts().sort_index())
print("Promedio:", df["review_score"].mean().round(3))

=== FILAS ===
orders original:         99441
orders entregados c/fecha: 96470
reviews original:        99224
reviews una por pedido:  98673
DataFrame final (df):    95824

=== VALIDACIONES ===
order_id único?          True
Solo pedidos 'delivered'? True
Nulos en fecha de entrega: 0
Nulos en review_score:     0
Nulos en customer_state:   0

=== TIPOS DE FECHA ===
order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
review_creation_date             datetime64[us]
dtype: object

=== PÉRDIDA DE DATOS ===
Pedidos conservados: 95,824 de 99,441 (96.4 %)
Pedidos descartados: 3,617

=== RETRASO ===
count    95824.00
mean       -11.91
std         10.11
min       -147.00
25%        -17.00
50%        -12.00
75%         -7.00
max        188.00
Name: dias_retraso, dtype: float64
Pedidos con retraso (>0): 6,381 (6.7 %)

=== CALIFICACIÓN ===
review_score
1     9353
2     2919
3     7916
4    18894
5    56742
Nam